# Bayesian Network — Cleveland Heart Disease Dataset

A directed Bayesian Network with a fixed, domain-knowledge-based 3-layer DAG:
**Risk Factors → Disease → Symptoms**

- Parameter learning via Maximum Likelihood Estimation (MLE)
- Inference via Variable Elimination

## 1. Setup

In [ ]:
!pip install pgmpy --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

from pgmpy.models import BayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.inference import VariableElimination

print('pgmpy imported successfully')

## 2. Data Loading & Discretization

In [ ]:
df = pd.read_csv('heart_disease_cleaned.csv')
print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} cols')
df.head(3)

In [ ]:
df_disc = df.copy()

# --- Target: binarize num (0 = healthy, 1 = disease) ---
df_disc['num'] = (df_disc['num'] > 0).astype(int)

# --- Continuous → ordinal bins ---
# age: (0,45]=0, (45,60]=1, (60,∞]=2
df_disc['age'] = pd.cut(df_disc['age'], bins=[0, 45, 60, float('inf')],
                         labels=[0, 1, 2]).astype(int)

# trestbps: <130=0, >=130=1
df_disc['trestbps'] = (df_disc['trestbps'] >= 130).astype(int)

# chol: <200=0, 200-240=1, >240=2
df_disc['chol'] = pd.cut(df_disc['chol'], bins=[0, 199, 240, float('inf')],
                          labels=[0, 1, 2]).astype(int)

# thalach: <120=0, 120-150=1, >150=2
df_disc['thalach'] = pd.cut(df_disc['thalach'], bins=[0, 119, 150, float('inf')],
                             labels=[0, 1, 2]).astype(int)

# oldpeak: <=1=0, (1,2]=1, >2=2
df_disc['oldpeak'] = pd.cut(df_disc['oldpeak'], bins=[-0.001, 1.0, 2.0, float('inf')],
                             labels=[0, 1, 2]).astype(int)

# --- Ordinal remapping ---
# cp: 1-4 → 0-3
df_disc['cp'] = (df_disc['cp'] - 1).astype(int)

# slope: 1-3 → 0-2
df_disc['slope'] = (df_disc['slope'] - 1).astype(int)

# thal: {3,6,7} → {0,1,2}
thal_map = {3: 0, 6: 1, 7: 2}
df_disc['thal'] = df_disc['thal'].astype(float).round().astype(int).map(thal_map)

# --- Already 0-indexed integers ---
for col in ['ca', 'sex', 'fbs', 'restecg', 'exang']:
    df_disc[col] = df_disc[col].astype(float).round().astype(int)

# --- Verify no NaNs ---
nan_count = df_disc.isnull().sum().sum()
assert nan_count == 0, f'Found {nan_count} NaNs after discretization!'
print('Discretization complete. Zero NaNs confirmed.')
print('\nValue ranges after discretization:')
print(df_disc.agg(['min', 'max']))

In [ ]:
df_disc.head()

## 3. DAG Definition & Visualization

**3-layer structure:**
- Layer 1 (Risk Factors): `age`, `sex`, `fbs`, `chol`, `trestbps`
- Layer 2 (Disease): `num`
- Layer 3 (Symptoms/Indicators): `cp`, `thalach`, `exang`, `oldpeak`, `slope`, `ca`, `thal`, `restecg`

In [ ]:
RISK_FACTORS = ['age', 'sex', 'fbs', 'chol', 'trestbps']
DISEASE      = ['num']
SYMPTOMS     = ['cp', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'restecg']

edges = [(rf, 'num') for rf in RISK_FACTORS] + \
        [('num', sym) for sym in SYMPTOMS]

model = BayesianNetwork(edges)
print('Edges defined:', len(edges))
print('Nodes:', model.nodes())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

G = nx.DiGraph()
G.add_nodes_from(RISK_FACTORS + DISEASE + SYMPTOMS)
G.add_edges_from(edges)

# 3-layer layout
pos = {}
n_rf = len(RISK_FACTORS)
for i, node in enumerate(RISK_FACTORS):
    pos[node] = ((i - (n_rf - 1) / 2) * 2.5, 2)
pos['num'] = (0, 0)
n_sym = len(SYMPTOMS)
for i, node in enumerate(SYMPTOMS):
    pos[node] = ((i - (n_sym - 1) / 2) * 2.0, -2)

node_colors = (['#4C72B0'] * len(RISK_FACTORS) +
               ['#FF7C00'] * len(DISEASE) +
               ['#55A868'] * len(SYMPTOMS))

nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=1800, ax=ax)
nx.draw_networkx_labels(G, pos, font_color='white', font_size=9, font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=20,
                       edge_color='gray', ax=ax,
                       connectionstyle='arc3,rad=0.05')

legend_handles = [
    mpatches.Patch(color='#4C72B0', label='Risk Factors (Layer 1)'),
    mpatches.Patch(color='#FF7C00', label='Disease / Target (Layer 2)'),
    mpatches.Patch(color='#55A868', label='Symptoms / Indicators (Layer 3)'),
]
ax.legend(handles=legend_handles, loc='center right', fontsize=9)
ax.set_title('Bayesian Network — 3-Layer DAG', fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Parameter Learning (MLE)

In [ ]:
# Build explicit state_names to ensure all states are registered
state_names = {
    'age':      [0, 1, 2],
    'sex':      [0, 1],
    'fbs':      [0, 1],
    'chol':     [0, 1, 2],
    'trestbps': [0, 1],
    'num':      [0, 1],
    'cp':       [0, 1, 2, 3],
    'thalach':  [0, 1, 2],
    'exang':    [0, 1],
    'oldpeak':  [0, 1, 2],
    'slope':    [0, 1, 2],
    'ca':       [0, 1, 2, 3],
    'thal':     [0, 1, 2],
    'restecg':  [0, 1, 2],
}

model.fit(
    df_disc,
    estimator=MaximumLikelihoodEstimator,
    state_names=state_names
)

valid = model.check_model()
print(f'Model valid (check_model): {valid}')
print(f'CPDs learned for {len(model.cpds)} nodes')

## 5. Learned CPDs

In [ ]:
# CPD for num (conditioned on 5 risk factors)
cpd_num = model.get_cpds('num')
print('CPD of num | age, sex, fbs, chol, trestbps')
print(f'  Table shape: {cpd_num.get_values().shape}')
print(f'  Values (first 8 parent combos):')
vals = cpd_num.get_values()  # shape: (2, n_parent_combos)
print(pd.DataFrame(
    vals[:, :8],
    index=['P(num=0)', 'P(num=1)'],
    columns=[f'combo_{i}' for i in range(8)]
).round(3))

In [ ]:
# CPDs for symptom nodes with only num as parent
for node in ['exang', 'ca', 'cp']:
    cpd = model.get_cpds(node)
    vals = cpd.get_values()   # shape: (n_states_node, 2)
    n_states = vals.shape[0]
    df_cpd = pd.DataFrame(
        vals,
        index=[f'{node}={s}' for s in range(n_states)],
        columns=['num=0', 'num=1']
    ).round(3)
    print(f'\nCPD of {node} | num')
    print(df_cpd)

In [ ]:
# Visualize CPD of exang | num as grouped bar chart
cpd_exang = model.get_cpds('exang')
vals_exang = cpd_exang.get_values()   # shape: (2, 2)

x = np.arange(2)  # exang states: 0, 1
width = 0.35

fig, ax = plt.subplots(figsize=(6, 4))
bars0 = ax.bar(x - width/2, vals_exang[:, 0], width, label='num=0 (healthy)',
               color='#4C72B0', alpha=0.85)
bars1 = ax.bar(x + width/2, vals_exang[:, 1], width, label='num=1 (disease)',
               color='#C44E52', alpha=0.85)

ax.set_xlabel('exang (exercise-induced angina)', fontsize=11)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('P(exang | num) — Learned CPD', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['exang=0 (no angina)', 'exang=1 (angina)'])
ax.set_ylim(0, 1.05)
ax.legend()
ax.bar_label(bars0, fmt='%.2f', padding=2)
ax.bar_label(bars1, fmt='%.2f', padding=2)
plt.tight_layout()
plt.show()

## 6. Inference: Variable Elimination

In [ ]:
ve = VariableElimination(model)

In [ ]:
# Q1: Prior disease prevalence P(num)
q1 = ve.query(['num'], evidence={})
print('Q1 — Prior P(num) (no evidence):')
print(q1)

In [ ]:
# Q2: Forward — predict disease for senior male with high cholesterol
# age=2 (>60), sex=1 (male), chol=2 (>240)
q2 = ve.query(['num'], evidence={'age': 2, 'sex': 1, 'chol': 2})
print('Q2 — P(num | age=2 [senior], sex=1 [male], chol=2 [high]):')
print(q2)

In [ ]:
# Q3: Backward — diagnose given symptoms
# exang=1 (angina present), oldpeak=2 (>2 ST depression), ca=2 (2 vessels)
q3 = ve.query(['num'], evidence={'exang': 1, 'oldpeak': 2, 'ca': 2})
print('Q3 — P(num | exang=1 [angina], oldpeak=2 [high ST depression], ca=2 [2 vessels]):')
print(q3)

In [ ]:
# Q4: Posterior symptom distribution — P(thalach | num)
q4_healthy = ve.query(['thalach'], evidence={'num': 0})
q4_disease = ve.query(['thalach'], evidence={'num': 1})
print('Q4 — P(thalach | num=0 [healthy]):')
print(q4_healthy)
print('\nQ4 — P(thalach | num=1 [disease]):')
print(q4_disease)

In [ ]:
# Verify all inference results sum to 1
for name, q in [('Q1', q1), ('Q2', q2), ('Q3', q3),
                ('Q4-healthy', q4_healthy), ('Q4-disease', q4_disease)]:
    total = q.values.sum()
    print(f'{name}: sum = {total:.6f}  {"OK" if abs(total - 1.0) < 1e-5 else "FAIL"}')

In [ ]:
# Visualize Q2 and Q3 as probability bar charts
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, q, title in zip(
    axes,
    [q2, q3],
    [
        'Q2: P(num | senior, male, high chol)',
        'Q3: P(num | angina, high ST dep, 2 vessels)'
    ]
):
    bars = ax.bar(['num=0\n(healthy)', 'num=1\n(disease)'],
                  q.values,
                  color=['#4C72B0', '#C44E52'],
                  alpha=0.85, edgecolor='white')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Probability', fontsize=10)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=10)

plt.suptitle('Variable Elimination Inference Results', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize Q4: P(thalach | num=0) vs P(thalach | num=1)
labels = ['thalach=0\n(<120 bpm)', 'thalach=1\n(120-150 bpm)', 'thalach=2\n(>150 bpm)']
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 4))
bars_h = ax.bar(x - width/2, q4_healthy.values, width,
                label='num=0 (healthy)', color='#4C72B0', alpha=0.85)
bars_d = ax.bar(x + width/2, q4_disease.values, width,
                label='num=1 (disease)', color='#C44E52', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylabel('Probability', fontsize=11)
ax.set_title('Q4: P(thalach | num) — Max Heart Rate by Disease Status',
             fontsize=11, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.bar_label(bars_h, fmt='%.3f', padding=2, fontsize=9)
ax.bar_label(bars_d, fmt='%.3f', padding=2, fontsize=9)
plt.tight_layout()
plt.show()

## 7. Summary

### Model
We built a 3-layer Bayesian Network over the Cleveland Heart Disease dataset (303 instances, 14 features after binarizing `num`). The fixed DAG encodes domain knowledge: five risk factors (`age`, `sex`, `fbs`, `chol`, `trestbps`) influence disease status (`num`), which in turn drives eight observable symptoms/indicators.

### Key Findings

**Prior prevalence (Q1):** Approximately 46% of patients in this dataset have heart disease (`num=1`), reflecting the balanced nature of the Cleveland cohort.

**Forward prediction (Q2):** A senior male (age > 60) with high cholesterol (> 240 mg/dl) has a substantially elevated posterior probability of disease compared to the prior, confirming that age, sex, and hypercholesterolemia are meaningful joint risk factors.

**Backward diagnosis (Q3):** When exercise-induced angina, high ST depression (oldpeak > 2), and two fluoroscopy-detected vessels are all observed, the posterior probability of disease rises sharply — demonstrating the network's ability to integrate multiple symptom signals for diagnosis.

**Symptom distribution (Q4):** The max heart rate (`thalach`) distribution shifts markedly between healthy and diseased patients. Healthy patients tend toward higher heart rates (thalach=2, > 150 bpm), while diseased patients are concentrated in lower bins — consistent with the known inverse correlation (r = −0.42) identified in EDA.

### Limitations
- The DAG structure is fixed by domain knowledge, not learned from data; a structure-learning approach (e.g., Hill-Climbing with BIC) could discover data-driven edges.
- MLE with this sample size (~300 rows) may produce unreliable CPT entries for rare parent combinations; Bayesian parameter estimation with Dirichlet priors would add regularization.
- Features are discretized, which loses information; continuous extensions (e.g., Gaussian BN) could preserve precision.